Gerando o banco de dados para o projeto

Passo 01: importar bibliotecas
Passo 02: definir as constantes
Passo 03: gerar as listas auxiliares
Passo 04: gerar o conteudo das colunas
Passo 05: gerar o dataframe
Passo 05: gerar o csv


In [ ]:
#Passo 01
import pandas as pd
import random 


#Passo 02

NumeroDeClientes = 2000
NumeroDeVendas = 200000

# Passo 03

Cidade = ['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Curitiba', 'Porto Alegre', 'Salvador', 'Fortaleza', 'Recife', 'Brasília',
            'Manaus', 'Belém', 'Goiânia', 'Campinas', 'São Luís', 'Maceió', 'Natal', 'Teresina', 'João Pessoa', 'Aracaju', 'Cuiabá', 'Campo Grande']

Categoria = ['Eletrônicos', 'Roupas', 'Alimentos', 'Móveis', 'Brinquedos', 'Esportes', 'Livros', 'Beleza', 'Automóveis', 'Jardinagem']

# Passo 04

cliente_cidade ={ }
for i in range(NumeroDeClientes):
    cliente_cidade[i + 1] = random.choice(Cidade)

produto_categoria = {}
for i in range(1, 601): #o segundo número não é incluído
    produto_categoria[i] = random.choice(Categoria)

dados = []
for i in range(NumeroDeVendas):
    id_cliente = random.randint(1, NumeroDeClientes)
    id_produto = random.randint(1, 600) #inclui o 1 e o 600
    venda = {
        'VendaID': i + 1,
        'ClienteID': id_cliente,
        'ProdutoID': id_produto,
        'DataVenda': pd.Timestamp(random.randint(2015, 2025), random.randint(1, 12), random.randint(1, 28)),
        'Categoria': produto_categoria[id_produto],
        'Cidade': cliente_cidade[id_cliente],
        'Quantidade': random.randint(1, 10),
        'Valor': round(random.uniform(10.0, 1000.0), 2)
    }
    dados.append(venda)

# Passo 05  

df_vendas = pd.DataFrame(dados)

#passo 06

print(df_vendas.head())
print(df_vendas.shape)
df_vendas.to_csv('vendas.csv', index=False)



In [1]:
import sqlite3
import pandas as pd
df = pd.read_csv('vendas.csv')
conn = sqlite3.connect('vendas.db')
df.to_sql('vendas', conn, if_exists='replace', index=False)
conn.close()
print("Dados exportados para vendas.db com sucesso!")

C:\Users\Dell\AppData\Local\Temp\ipykernel_17968\2675088442.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Dados exportados para vendas.db com sucesso!


Adicionando os index 

In [2]:
import sqlite3

conn = sqlite3.connect('vendas.db')

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_cliente_data
ON vendas (ClienteID, DataVenda);
""")

conn.commit()

print("Índice criado com sucesso!")

Índice criado com sucesso!


VERSÃO 01 : SQL tradicional 

In [3]:
import pandas as pd
import sqlite3
import time
conn = sqlite3.connect('vendas.db')

In [ ]:


query = """

SELECT CLIENTEID, (
SELECT  AVG(VALOR) from (

SELECT VALOR FROM 
vendas v2 WHERE v2.CLIENTEID = v1.CLIENTEID
 ORDER BY DATAVENDA DESC LIMIT 3) ultimas_3) Media
FROM (
    SELECT DISTINCT ClienteID
    FROM vendas
) v1
ORDER BY CLIENTEID
"""
tempos = []

for _ in range(3):
    inicio = time.perf_counter()

    result = pd.read_sql_query(query, conn)

    fim = time.perf_counter()

    tempos.append(fim - inicio)

print("Tempos:", tempos)
print("Tempo médio:", sum(tempos)/len(tempos))

display(result)


Tempos: [0.05719550000003437, 0.045206200000166064, 0.04651190000004135]
Tempo médio: 0.049637866666747264


,ClienteID,Media
0,1,648.763333
1,2,377.450000
2,3,424.076667
3,4,372.890000
4,5,609.143333
...,...,...
1995,1996,708.210000
1996,1997,343.303333
1997,1998,404.270000
1998,1999,572.616667


In [16]:
query = """
EXPLAIN QUERY PLAN

SELECT CLIENTEID, (
    SELECT AVG(VALOR)
    FROM (
        SELECT VALOR
        FROM vendas v2
        WHERE v2.CLIENTEID = v1.CLIENTEID
        ORDER BY DATAVENDA DESC
        LIMIT 3
    ) ultimas_3
) Media
FROM (
    SELECT DISTINCT ClienteID
    FROM vendas
) v1
ORDER BY CLIENTEID
"""

pd.read_sql_query(query, conn)

,id,parent,notused,detail
0,2,0,0,CO-ROUTINE v1
1,5,2,212,SCAN vendas USING COVERING INDEX idx_cliente_data
2,14,0,186,SCAN v1
3,18,0,0,CORRELATED SCALAR SUBQUERY 2
4,21,18,0,CO-ROUTINE ultimas_3
5,26,21,62,SEARCH v2 USING INDEX idx_cliente_data (Client...
6,41,18,32,SCAN ultimas_3
7,57,0,0,USE TEMP B-TREE FOR ORDER BY


VERSÃO 02 : Window Function (intermediário)

In [13]:


query = """

SELECT CLIENTEID, AVG(Valor) AS Media FROM (
SELECT
    ClienteID,
    Valor,
    ROW_NUMBER() OVER (
        PARTITION BY ClienteID
        ORDER BY DataVenda DESC
    ) AS ordem
FROM vendas) vendas_ordenadas
WHERE ordem <= 3
GROUP BY ClienteID

"""
tempos = []

for _ in range(3):
    inicio = time.perf_counter()

    result = pd.read_sql_query(query, conn)

    fim = time.perf_counter()

    tempos.append(fim - inicio)

print("Tempos:", tempos)
print("Tempo médio:", sum(tempos)/len(tempos))

display(result)


Tempos: [1.1872714999999516, 1.1933645999999953, 1.2213447999999971]
Tempo médio: 1.2006602999999814


,ClienteID,Media
0,1,648.763333
1,2,377.450000
2,3,424.076667
3,4,372.890000
4,5,609.143333
...,...,...
1995,1996,708.210000
1996,1997,343.303333
1997,1998,404.270000
1998,1999,572.616667


In [14]:
query = """
EXPLAIN QUERY PLAN
SELECT CLIENTEID, AVG(Valor) AS Media FROM (
SELECT
    ClienteID,
    Valor,
    ROW_NUMBER() OVER (
        PARTITION BY ClienteID
        ORDER BY DataVenda DESC
    ) AS ordem
FROM vendas) vendas_ordenadas
WHERE ordem <= 3
GROUP BY ClienteID
"""

pd.read_sql_query(query, conn)

,id,parent,notused,detail
0,2,0,0,CO-ROUTINE vendas_ordenadas
1,5,2,0,CO-ROUTINE (subquery-3)
2,9,5,224,SCAN vendas USING INDEX idx_cliente_data
3,27,5,0,USE TEMP B-TREE FOR LAST TERM OF ORDER BY
4,48,2,216,SCAN (subquery-3)
5,98,0,216,SCAN vendas_ordenadas
6,103,0,0,USE TEMP B-TREE FOR GROUP BY


In [6]:


query = """

SELECT CLIENTEID, (
SELECT  AVG(VALOR) from (

SELECT VALOR FROM vendas v2 WHERE v2.CLIENTEID = v1.CLIENTEID ORDER BY DATAVENDA DESC LIMIT 3) ultimas_3) Media
FROM (
    SELECT DISTINCT ClienteID
    FROM vendas
) v1
ORDER BY CLIENTEID
"""
tempos = []

for _ in range(3):
    inicio = time.perf_counter()

    result = pd.read_sql_query(query, conn)

    fim = time.perf_counter()

    tempos.append(fim - inicio)

print("Tempos:", tempos)
print("Tempo médio:", sum(tempos)/len(tempos))

display(result)


Tempos: [0.05531710000013845, 0.040659399999867674, 0.041281000000026324]
Tempo médio: 0.04575250000001082


,ClienteID,Media
0,1,648.763333
1,2,377.450000
2,3,424.076667
3,4,372.890000
4,5,609.143333
...,...,...
1995,1996,708.210000
1996,1997,343.303333
1997,1998,404.270000
1998,1999,572.616667


VERSÃO 03 : Window Function + índice (Avançado)

In [ ]:
pd.read_sql_query("""
EXPLAIN QUERY PLAN
SELECT CLIENTEID, (
SELECT AVG(VALOR)
FROM (
SELECT VALOR
FROM vendas v2
WHERE v2.CLIENTEID = v1.CLIENTEID
ORDER BY DATAVENDA DESC
LIMIT 3
) ultimas_3) Media
FROM (
SELECT DISTINCT ClienteID
FROM vendas
) v1
ORDER BY CLIENTEID;
""", conn)